# model3.0.ipynb — LSTM for Flotation Concentrate Quality Prediction

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

from src.models.lstm import FloatLSTM, AssayDataset, train_lstm_model
from src.utils.visualization import plot_training_curve, plot_actual_vs_predicted
from src.utils.metrics import print_evaluation_metrics

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
df = pd.read_csv('../data/Data.csv', parse_dates=[0], low_memory=False)
date_col = df.columns[0]
df = df.rename(columns={date_col: 'date'})
numeric_cols = df.columns.drop('date')
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df = df.sort_values('date').reset_index(drop=True)

target_cols = ['% Iron Concentrate', '% Silica Concentrate']
sensor_cols = [c for c in df.columns if c not in ['date'] + target_cols]
mask = (df[target_cols].diff().abs() > 0).any(axis=1)
df_real = df[mask].copy().reset_index(drop=True)
print(f'Real measurement rows: {len(df_real):,}')

In [ ]:
SEQ_LEN = 60
X_raw = df_real[sensor_cols].values.astype(np.float32)
y_raw = df_real[target_cols].values.astype(np.float32)
valid_mask = ~(np.isnan(X_raw).any(axis=1) | np.isnan(y_raw).any(axis=1))
X_raw, y_raw = X_raw[valid_mask], y_raw[valid_mask]

split_idx = int(len(X_raw) * 0.8)
X_train_raw, X_test_raw = X_raw[:split_idx], X_raw[split_idx:]
y_train_raw, y_test_raw = y_raw[:split_idx], y_raw[split_idx:]

feat_scaler = StandardScaler()
target_scaler = StandardScaler()
X_train_sc = feat_scaler.fit_transform(X_train_raw)
X_test_sc = feat_scaler.transform(X_test_raw)
y_train_sc = target_scaler.fit_transform(y_train_raw)
y_test_sc = target_scaler.transform(y_test_raw)

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len : i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_tr_seq, y_tr_seq = make_sequences(X_train_sc, y_train_sc, SEQ_LEN)
X_te_seq, y_te_seq = make_sequences(X_test_sc,  y_test_sc,  SEQ_LEN)

In [ ]:
train_loader = DataLoader(AssayDataset(X_tr_seq, y_tr_seq), batch_size=64, shuffle=False)
test_loader = DataLoader(AssayDataset(X_te_seq, y_te_seq), batch_size=64, shuffle=False)

n_features = X_tr_seq.shape[2]
model = FloatLSTM(input_size=n_features, hidden_size=128, num_layers=2, output_size=2).to(DEVICE)
model, train_losses, val_losses = train_lstm_model(model, train_loader, test_loader, epochs=150, device=DEVICE)

plot_training_curve(train_losses, val_losses, 'LSTM Training Curve')

In [ ]:
model.eval()
preds_sc = []
with torch.no_grad():
    for Xb, _ in test_loader:
        preds_sc.append(model(Xb.to(DEVICE)).cpu().numpy())

y_pred_sc = np.vstack(preds_sc)
y_pred = target_scaler.inverse_transform(y_pred_sc)
y_true = target_scaler.inverse_transform(y_te_seq)

print_evaluation_metrics(y_true, y_pred, target_cols)
plot_actual_vs_predicted(y_true, y_pred, target_cols, 'LSTM — Actual vs Predicted')